In [ ]:
# goal is to create an mnist data reader with pure NumPy
# Chooose the layout of the model i think i will go with 3 blue 1 brown showcase from YT
# LETS GO
# input is 28x28 therefore first layer is 784 x 1 vecotor
#

In [113]:
import numpy as np
import pandas as pd

df = pd.read_csv("/content/sample_data/mnist_train_small.csv")
df2 = pd.read_csv("/content/sample_data/mnist_test.csv")

y_train = df.iloc[:,0]
x_train = df.iloc[:,1:]
print(y_train.shape)
print(x_train.shape)


y_test = df2.iloc[:,0]
x_test = df2.iloc[:,1:]
y_testn = y_test.to_numpy()
x_testn = x_test.to_numpy()
x_testn = (x_testn/255).astype(np.float64)




label = y_train.to_numpy()
input = x_train.to_numpy()
input = (input/255).astype(np.float64)
print(y_train.head())
print(input[0].max())



(19999,)
(19999, 784)
0    5
1    7
2    9
3    5
4    2
Name: 6, dtype: int64
1.0


In [114]:
# initializing the Neural net

input_layer = np.random.rand(784,1) # for now
# hidden1
W1 = np.random.rand(128,784) - 0.5
B1 = np.zeros((128,1))
#hidden2
W2 = np.random.rand(64,128) - 0.5
B2 = np.zeros((64,1))
#lastLayer
W3 = np.random.rand(10,64) - 0.5
B3 = np.zeros((10,1))

# output is 10,1 vector
W1.shape

# so it will be W1 applied to input W2 applied to the out vector of 1 layers .... to the last (ignoring bias) shapes are good mb change later if training is slow

(128, 784)

In [115]:
def forward(input_layer,W,B):
  output = np.matmul(W, input_layer)
  z = np.add(output, B)
  a = Relu(z)
  return a, z


def Relu(input_layer):
  output = np.maximum(0, input_layer)
  return output

def softmax(input_layer):
  max = np.max(input_layer)
  output = np.exp(input_layer - max)

  return output / np.sum(output)


def cross_entropy(input_layer, expected_output):
  log_in = np.log(input_layer)
  loss = -np.dot(log_in.T, expected_output)
  return loss.squeeze()

def forward_last(input, W, b):
  output = np.matmul(W, input)
  z = np.add(output,b)
  a = softmax(z)
  return a, z






In [116]:
def last_grad(prediction, previous_layer, target): # so the partial derivative dc0/dWl = (alk - yk) * ak
  dCdz = prediction - target
  grad_mat = np.outer(dCdz, previous_layer) # multiplied by last activation

  grad_b = dCdz # for bias

  return grad_mat , grad_b , dCdz


def grad(delta, W_prev, a_next, z_cur):
  dadz = (z_cur > 0).astype(float) # this is da(l-1)/dz(l-1)
  dcda = np.matmul(W_prev.T, delta) # this is dc/da(l-1)
  dcdz = np.multiply(dcda, dadz) # this is dc/dz(l-1)
  grad_mat = np.matmul(dcdz,a_next.T) # this is dc/dw(l-1)

  grad_b = dcdz

  return grad_mat , grad_b , dcdz

def optimizer_step(W1, b1, W2, b2, W3, b3, jW1, jb1, jW2, jb2, jW3,  jb3, lr):
  W1 = W1- lr* jW1
  b1 = b1 - lr* jb1

  W2 = W2- lr* jW2
  b2 = b2 - lr* jb2

  W3 = W3- lr* jW3
  b3 = b3 - lr* jb3

  return W1, b1, W2, b2, W3, b3




In [121]:

def training(inputs, labels):
  global W1, B1, W2, B2, W3, B3

  epochs = 3
  lr = 0.1
  batch_n = 32

  n = len(inputs)

  counter = 0


  for epoch in range(epochs):
    for start in range(0,n, batch_n):
      end = min(start + batch_n, n)
      batch = inputs[start:end]
      y = labels[start:end]


      jw1 = np.zeros_like(W1); jb1 = np.zeros_like(B1)
      jw2 = np.zeros_like(W2); jb2 = np.zeros_like(B2)
      jw3 = np.zeros_like(W3); jb3 = np.zeros_like(B3)

      for j in range (len(batch)):
        label = y[j]

        # forward
        x = batch[j]
        x = np.expand_dims(x,axis=1)

        v_label = np.zeros((10,1))
        v_label[label] = 1


        a_l2, z_l2 = forward(x, W1, B1)

        a_l1, z_l1 = forward(a_l2 , W2, B2)

        a_l, z_l   = forward_last(a_l1 , W3, B3)


        #grad
        jw3_eg , jb3_eg , dz_l = last_grad(a_l,a_l1, v_label)

        jw2_eg , jb2_eg , dz_l1 = grad(dz_l, W3, a_l2, z_l1)

        jw1_eg , jb1_eg , dz_l2 = grad(dz_l1, W2, x, z_l2)

        # acc_grad
        jw1 += jw1_eg
        jw2 += jw2_eg
        jw3 += jw3_eg

        jb1 += jb1_eg
        jb2 += jb2_eg
        jb3 += jb3_eg

      #avg the jacobians
      jw1 /= len(batch)
      jw2 /= len(batch)
      jw3 /= len(batch)

      jb1 /= len(batch)
      jb2 /= len(batch)
      jb3 /= len(batch)

      W1, B1, W2, B2, W3, B3 = optimizer_step(W1, B1, W2, B2, W3, B3, jw1, jb1, jw2, jb2, jw3,  jb3, lr)


    correct = 0
    for i in range(len(inputs)):
      c = inputs[i]
      inp = np.expand_dims(c,axis=1)
      y = labels[i]



      a1,a = forward(inp, W1, B1)
      a2,b = forward(a1 , W2, B2)
      output,c = forward_last(a2 , W3, B3)


      pred = np.argmax(output)


      if pred == y:
        correct+=1
    accuracy = correct/(len(inputs))
    print("accuracy is :" ,accuracy)









In [118]:
def validate(inputs, labels):
  global W1, B1, W2, B2, W3, B3



  correct = 0
  for i in range(len(inputs)):
    c = inputs[i]
    inp = np.expand_dims(c,axis=1)
    y = labels[i]



    a1,a = forward(inp, W1, B1)
    a2,b = forward(a1 , W2, B2)
    output,c = forward_last(a2 , W3, B3)


    pred = np.argmax(output)


    if pred == y:
      correct+=1
  accuracy = correct/(len(inputs))
  print("accuracy is :" ,accuracy)

In [122]:
training(input, label)


accuracy is : 0.9989999499974999
accuracy is : 0.9991999599979999
accuracy is : 0.99949997499875


In [123]:
validate(x_testn, y_testn)

accuracy is : 0.9512951295129513
